# 05 — Library WMF (implicit ALS) + Surprise SVD

Benchmarks against two standard libraries on the **exact same** candidates/metrics as the
baseline:
- **`implicit.AlternatingLeastSquares`** — the canonical WMF for implicit feedback (the
  library implementation of Hu/Koren/Volinsky 2008).
- **`surprise.SVD`** — classical Funk-SVD on the explicit ratings, as a sanity comparison.

These are the numbers quoted on the resume.

In [1]:
import os, sys, time
os.environ['OPENBLAS_NUM_THREADS'] = '1'
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, scipy.sparse as sp
import recsys_utils as ru
ART = '../artifacts'
train_mat = sp.load_npz(f'{ART}/train_mat.npz')
c = np.load(f'{ART}/cands.npz'); users, cands = c['users'], c['cands']
n_users, n_items = train_mat.shape
print(f'{n_users:,} users | {n_items:,} items | {len(users):,} eval users')

162,541 users | 55,413 items | 162,495 eval users


## implicit ALS (Weighted Matrix Factorization)

`factors=64` latent dimensions, `alpha=40` confidence scaling, L2 `regularization=0.05`,
15 ALS sweeps. ALS alternates: fix item vectors → solve each user vector in closed form,
then fix users → solve each item. No learning rate, and each sweep is embarrassingly parallel.

In [2]:
from implicit.cpu.als import AlternatingLeastSquares
t = time.time()
als = AlternatingLeastSquares(factors=64, regularization=0.05, alpha=ru.ALPHA,
                              iterations=15, random_state=42)
als.fit(train_mat, show_progress=False)
print(f'trained implicit ALS in {time.time()-t:.1f}s')
als_scores = ru.score_als(als.user_factors, als.item_factors, users, cands)
r_als, n_als = ru.score_metrics(als_scores, k=10)
print(f'implicit ALS (WMF)  ->  Recall@10 = {r_als:.4f}   NDCG@10 = {n_als:.4f}')

/home/racloop/Documents/Personal/coursera/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


trained implicit ALS in 27.2s


implicit ALS (WMF)  ->  Recall@10 = 0.9714   NDCG@10 = 0.8097


## Surprise SVD (classical Funk-SVD on explicit ratings)

Trained on a capped sample of the explicit ratings (Surprise is single-threaded and not
built for 25M rows). Evaluated on the same held-out candidates by predicting a score for
each (user, candidate-movie) pair.

In [3]:
from surprise import SVD, Dataset, Reader
import pandas as pd
# reconstruct (u,i,r) triples from the training matrix, capped for Surprise's speed
coo = train_mat.tocoo()
tr = pd.DataFrame({'u': coo.row, 'i': coo.col, 'r': coo.data})
SAMPLE = 2_000_000
if len(tr) > SAMPLE:
    tr = tr.sample(SAMPLE, random_state=0)
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(tr[['u','i','r']], reader).build_full_trainset()
t = time.time()
svd = SVD(n_factors=64, n_epochs=20, random_state=42)
svd.fit(data)
print(f'trained Surprise SVD on {len(tr):,} ratings in {time.time()-t:.1f}s')

trained Surprise SVD on 2,000,000 ratings in 19.1s


In [4]:
# score candidates with SVD (uses the raw inner-id estimates; unknown ids fall back to global mean)
def svd_score_row(u, row):
    return np.array([svd.predict(u, int(it)).est for it in row], dtype=np.float32)
svd_scores = np.empty(cands.shape, dtype=np.float32)
for k in range(len(users)):
    svd_scores[k] = svd_score_row(int(users[k]), cands[k])
r_svd, n_svd = ru.score_metrics(svd_scores, k=10)
print(f'Surprise SVD        ->  Recall@10 = {r_svd:.4f}   NDCG@10 = {n_svd:.4f}')

Surprise SVD        ->  Recall@10 = 0.5144   NDCG@10 = 0.3067


## Results summary

In [5]:
item_pop = np.asarray(train_mat.getnnz(axis=0)).ravel().astype(np.float32)
r_pop, n_pop = ru.score_metrics(ru.score_popularity(item_pop, cands), k=10)
print(f'{"Model":<22}{"Recall@10":>12}{"NDCG@10":>12}')
print('-'*46)
for name, r, n in [('Popularity baseline', r_pop, n_pop),
                   ('Surprise SVD', r_svd, n_svd),
                   ('implicit ALS (WMF)', r_als, n_als)]:
    print(f'{name:<22}{r:>12.4f}{n:>12.4f}')

Model                    Recall@10     NDCG@10
----------------------------------------------
Popularity baseline         0.9251      0.6606
Surprise SVD                0.5144      0.3067
implicit ALS (WMF)          0.9714      0.8097
